# Instructor worked practice examples
Run from the repository root. These are possible solutions for `practice-1-guided.ipynb` and `practice-2-eda.ipynb`, rather than additional material to work through in the main teaching notebooks.

Keep this file separate while students work. See `INSTRUCTOR_GUIDE.md` for discussion notes.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

# Open this notebook from the visualization_demo folder.
gapminder = pd.read_csv("data/gapminder.csv")
snapshot = gapminder.loc[gapminder["year"].eq(2007)].copy()
continents = ["Africa", "Americas", "Asia", "Europe", "Oceania"]
palette = dict(zip(continents, ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#9A6700"]))
plt.rcdefaults()
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

import seaborn as sns


## Matplotlib revision


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
ax.scatter(snapshot["gdpPercap"], snapshot["lifeExp"], color="#0072B2", s=38, alpha=0.75)
ax.set_xscale("log")
ax.set(xlabel="GDP per capita (inflation-adjusted US$); log scale", ylabel="Life expectancy at birth (years)",
       title="Higher GDP generally accompanies longer life expectancy, 2007")
ax.set_xticks([1000, 10000, 100000], labels=["1,000", "10,000", "100,000"])
ax.grid(axis="y", alpha=0.18)
Path("outputs").mkdir(exist_ok=True)
fig.savefig("outputs/instructor_scatter.png", dpi=200, bbox_inches="tight")
plt.show()


One country is one point. Life expectancy tends to be higher among countries with greater GDP per capita, with substantial variation at similar GDP levels. These country-level observational data do not establish a causal effect of GDP.


## Seaborn distribution comparison


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
sns.boxplot(data=snapshot, x="continent", y="lifeExp", order=continents,
            color="lightgray", showfliers=False, ax=ax)
np.random.seed(386)
sns.stripplot(data=snapshot, x="continent", y="lifeExp", order=continents,
              jitter=0.18, color="#333333", alpha=0.65, size=4, ax=ax)
ax.set(xlabel="Continent", ylabel="Life expectancy (years)",
       title="Life expectancy varies within and across continents, 2007")
plt.show()


A mean alone hides overlap, spread, and unusual observations. The points also reveal the uneven numbers of countries per group. Oceania contains only two countries in this extract; do not generalize from its boxplot to every place in the region.


## Plotly investigation: numeric cross-check


In [ ]:
comparison = snapshot.loc[snapshot["country"].isin(["Vietnam", "Yemen, Rep."]),
                          ["country", "gdpPercap", "lifeExp"]]
comparison.round(1)


Vietnam and Yemen have relatively similar GDP per capita in this snapshot but visibly different life expectancy. Possible follow-up: how do health care, disease burdens, conflict, education, or measurement differences relate to that gap? None of these explanations is established by this plot.

The Plotly notebook itself provides working hover, bubbles, animation, and HTML export examples. This numeric cross-check does not test interactive rendering.


## EDA extension: largest endpoint changes


In [ ]:
endpoints = gapminder.loc[gapminder["year"].isin([1952, 2007])].pivot(
    index="country", columns="year", values="lifeExp")
changes = endpoints.assign(change=endpoints[2007] - endpoints[1952])
top_changes = changes.nlargest(10, "change").sort_values("change")
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
ax.barh(top_changes.index, top_changes["change"], color="#0072B2")
ax.set(xlabel="Change in life expectancy at birth (years)", ylabel="Country",
       title="Ten largest increases from 1952 to 2007 in the Gapminder extract")
plt.show()


Each bar compares the same country at two endpoints, using only countries in this teaching extract. This selected top-ten ranking hides intermediate changes and countries outside the extract. It does not identify what caused the increases.


## Tool-choice discussion
- Precise static layout: Matplotlib offers direct control.
- Group distribution comparison: Seaborn supplies concise statistical summaries and DataFrame mappings, with Matplotlib available for refinement.
- Hover inspection: Plotly Express supplies browser interaction with a small plotting call.
- A dropdown that triggers Python data filtering: Streamlit supplies the app control and execution flow; Plotly can supply the interactive chart displayed by the app.

Other choices can be defensible when the student explains the task, audience, and sharing requirements.
